In [1]:
!pip install -q codecarbon


In [4]:
from datasets import load_dataset
from codecarbon import EmissionsTracker
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

SEED = 42

def load_data(n_train=10_000, n_test=2_000, seed=SEED):
    ds = load_dataset("fancyzhx/ag_news")
    train = ds["train"].shuffle(seed=seed).select(range(n_train))
    test = ds["test"].shuffle(seed=seed).select(range(n_test))
    return train, test

def mesurer(nom, fonction_entrainement):
    tracker = EmissionsTracker(
        project_name=nom,
        output_file="emissions.csv",
        measure_power_secs=1,   # mesure chaque seconde (entraînement court)
        log_level="error",
    )
    tracker.start()
    accuracy = fonction_entrainement()
    kg = tracker.stop()
    print(f"{nom} | accuracy={accuracy:.4f} | {kg*1000:.4f} g CO2eq")
    return accuracy, kg

train, test = load_data()   # téléchargement HORS mesure

def train_light():
    vectorizer = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2), sublinear_tf=True)
    X_train = vectorizer.fit_transform(train["text"])
    X_test = vectorizer.transform(test["text"])
    clf = LogisticRegression(max_iter=1000, random_state=SEED)
    clf.fit(X_train, train["label"])
    return accuracy_score(test["label"], clf.predict(X_test))

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [5]:
for i in range(1, 4):
    mesurer(f"leger_run{i}", train_light)

[codecarbon WARNING @ 13:21:26] Multiple instances of codecarbon are allowed to run at the same time.


leger_run1 | accuracy=0.8840 | 0.0114 g CO2eq
leger_run2 | accuracy=0.8840 | 0.0072 g CO2eq
leger_run3 | accuracy=0.8840 | 0.0083 g CO2eq


In [7]:
import os

# On repart d'un CSV vide (sinon les nouvelles lignes s'ajoutent aux anciennes)
if os.path.exists("emissions.csv"):
    os.remove("emissions.csv")

# Échauffement : on entraîne une fois SANS mesurer
train_light()

# 5 runs mesurés
resultats = [mesurer(f"leger_run{i}", train_light) for i in range(1, 6)]

print("\nAccuracies :", [round(acc, 4) for acc, kg in resultats])

leger_run1 | accuracy=0.8840 | 0.0070 g CO2eq
leger_run2 | accuracy=0.8840 | 0.0078 g CO2eq
leger_run3 | accuracy=0.8840 | 0.0081 g CO2eq
leger_run4 | accuracy=0.8840 | 0.0086 g CO2eq
leger_run5 | accuracy=0.8840 | 0.0133 g CO2eq

Accuracies : [0.884, 0.884, 0.884, 0.884, 0.884]


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [8]:
import pandas as pd
df = pd.read_csv("emissions.csv")
df["emissions_g"] = df["emissions"] * 1000
cols = ["duration", "energy_consumed", "emissions_g"]
print(df[["project_name"] + cols])
print("\nMoyenne :\n", df[cols].mean())
print("\nVariation (%) :\n", (df[cols].std() / df[cols].mean() * 100).round(1))
print("\nInfos matériel :")
print(df[["cpu_model", "ram_total_size", "country_name", "region"]].iloc[0])


  project_name   duration  energy_consumed  emissions_g
0   leger_run1   9.831207         0.000042     0.007020
1   leger_run2  10.452492         0.000047     0.007812
2   leger_run3  10.775493         0.000049     0.008099
3   leger_run4  11.831039         0.000052     0.008571
4   leger_run5  17.327506         0.000080     0.013257

Moyenne :
 duration           12.043548
energy_consumed     0.000054
emissions_g         0.008952
dtype: float64

Variation (%) :
 duration           25.3
energy_consumed    27.6
emissions_g        27.6
dtype: float64

Infos matériel :
cpu_model         Intel(R) Xeon(R) CPU @ 2.20GHz
ram_total_size                         12.671406
country_name                       United States
region                                    oregon
Name: 0, dtype: object
